# Spatial-Cell Shapley Audit Benchmark

Ноутбук для classifier-only audit-benchmark на реальной модели и реальном датасете. Он сравнивает `IG`, `NAA` и один фиксированный вариант `Cheap-IG` по согласованности с `Monte-Carlo Shapley` oracle на пуле `spatial cells` выбранного слоя.

Текущая постановка:
- модель: `yolo11s-cls`
- слой: `model.6`
- unit mode: `spatial_cell`
- target: clean top-1 logit
- pool size: `196` (полная spatial grid `14x14` для `model.6`)
- permutations: `128`
- pool selection: `active_random` с возможностью переключиться на `stratified_activation_change`
- oracle imputer: `black_act`
- методы: `IG`, `NAA`, `Cheap-IG+[0,0.1]/k8000/zero`


## Импорты

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from modules.neuron_shapley_benchmark import (
    benchmark_classifier_neuron_shapley,
    default_classifier_method_specs,
    render_neuron_shapley_report,
)


## Параметры

In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 30

CLASSIFIER_LAYER = "model.6"
UNIT_MODE = "spatial_cell"
N_STEPS = 128
POOL_SIZE = 196
NUM_PERMUTATIONS = 128
POOL_SELECTION_MODE = "active_random"
ACTIVE_MIN_ABS_DELTA = 1e-6
STRATIFIED_NUM_BINS = 4
ORACLE_IMPUTER_KIND = "black_act"
RANDOM_SEED = 0
PREVIEW_IMAGES = 5
NDCG_K = 10
RECALL_K = 10
SIGN_THRESHOLD_RATIO = 0.05
CLEAR_EVERY = 8
FD_EPS = 1e-3
BLUR_SIGMA = 16.0

CACHE_ROOT = Path("output/neuron_shapley_cache")
OUTPUT_DIR = Path("output/neuron_shapley_audit_oxford_pets_30_spatial_cell")
REFRESH_CORE = False
REFRESH_ORACLE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


## Датасет

In [3]:
def collect_image_paths(root: Path, n_images: int):
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    paths = sorted(
        [path for path in root.iterdir() if path.suffix.lower() in exts],
        key=lambda path: path.name.lower(),
    )
    return [str(path) for path in paths[:n_images]]


IMAGE_PATHS = collect_image_paths(OXFORD_PETS_DIR, N_IMAGES)
len(IMAGE_PATHS), IMAGE_PATHS[:3]


(30,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg'])

## Методы

In [4]:
METHOD_SPECS = default_classifier_method_specs()
METHOD_SPECS


[{'kind': 'ig', 'segment_start': 0.0, 'segment_end': 1.0, 'name': 'IG'},
 {'kind': 'naa', 'segment_start': 0.0, 'segment_end': 1.0, 'name': 'NAA'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.1,
  'selection_mode': 'positive',
  'selection_top_k': 8000,
  'fill_mode': 'zero',
  'fill_rho': 0.8,
  'name': 'Cheap-IG+[0,0.1]/k8000/zero'}]

## Запуск Benchmark

In [5]:
results = benchmark_classifier_neuron_shapley(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    unit_mode=UNIT_MODE,
    n_steps=N_STEPS,
    pool_size=POOL_SIZE,
    num_permutations=NUM_PERMUTATIONS,
    pool_selection_mode=POOL_SELECTION_MODE,
    active_min_abs_delta=ACTIVE_MIN_ABS_DELTA,
    stratified_num_bins=STRATIFIED_NUM_BINS,
    oracle_imputer_kind=ORACLE_IMPUTER_KIND,
    random_seed=RANDOM_SEED,
    preview_images=PREVIEW_IMAGES,
    ndcg_k=NDCG_K,
    recall_k=RECALL_K,
    sign_threshold_ratio=SIGN_THRESHOLD_RATIO,
    blur_sigma=BLUR_SIGMA,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_oracle=REFRESH_ORACLE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
    verbose=False,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


output_dir: /Users/ashentide/PycharmProjects/PaperImplementations/output/neuron_shapley_audit_oxford_pets_30_spatial_cell
report_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/neuron_shapley_audit_oxford_pets_30_spatial_cell/neuron_shapley_report.md
summary_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/neuron_shapley_audit_oxford_pets_30_spatial_cell/neuron_shapley_summary.json


## Перерисовка отчёта и figure-артефактов

In [6]:
artifacts = render_neuron_shapley_report(results, output_dir=OUTPUT_DIR)
artifacts["report_path"], artifacts["summary_path"]


('output/neuron_shapley_audit_oxford_pets_30_spatial_cell/neuron_shapley_report.md',
 'output/neuron_shapley_audit_oxford_pets_30_spatial_cell/neuron_shapley_summary.json')

## Markdown-отчёт

In [7]:
display(Markdown(Path(artifacts["report_path"]).read_text(encoding="utf-8")))


# Neuron Shapley Audit Benchmark

Classifier-only Monte-Carlo Shapley audit benchmark on real model and dataset.

## Configuration

- layer_name=`model.6`
- unit_mode=`spatial_cell`
- n_steps=`128`
- pool_size=`196`
- num_permutations=`128`
- pool_selection_mode=`active_random`
- active_min_abs_delta=`1e-06`
- stratified_num_bins=`4`
- oracle_imputer_kind=`black_act`
- random_seed=`0`
- n_images=`30`

## Summary Table

| Method | Spearman | NDCG@k | Recall@k | Sign agreement | runtime_s | benchmark_runtime_s | abs_error |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| NAA | 0.5934 | 0.8273 | 0.6100 | 0.7997 | 2.8264 | 2.8281 | 10.9828 |
| IG | 0.5505 | 0.8294 | 0.6567 | 0.8353 | 5.4657 | 5.4678 | 0.8997 |
| Cheap-IG+[0,0.1]/k8000/zero | 0.4208 | 0.7420 | 0.5867 | 0.8203 | 2.3483 | 2.3496 | 108.6371 |

![](output/neuron_shapley_audit_oxford_pets_30_spatial_cell/figures/summary_metrics.png)

![](output/neuron_shapley_audit_oxford_pets_30_spatial_cell/figures/distribution_metrics.png)

![](output/neuron_shapley_audit_oxford_pets_30_spatial_cell/figures/pairwise_spearman.png)

## Visual Preview

Page 1 | methods: `['IG', 'NAA', 'Cheap-IG+[0,0.1]/k8000/zero']`

![](output/neuron_shapley_audit_oxford_pets_30_spatial_cell/figures/preview_page_1.png)



## Leaderboard

In [8]:
summary_df = pd.DataFrame(results["summary"]["summary_rows"])
summary_df = summary_df[
    [
        "method_name",
        "spearman_mean",
        "spearman_std",
        "ndcg_at_k_mean",
        "ndcg_at_k_std",
        "recall_at_k_mean",
        "recall_at_k_std",
        "sign_agreement_mean",
        "sign_agreement_std",
        "runtime_s_mean",
        "benchmark_runtime_s_mean",
        "method_abs_error_mean",
        "selected_neurons_mean",
        "n_images",
    ]
]
summary_df = summary_df.rename(
    columns={
        "method_name": "method",
        "spearman_mean": "spearman_mean",
        "spearman_std": "spearman_std",
        "ndcg_at_k_mean": f"ndcg@{results['summary']['ndcg_k']}_mean",
        "ndcg_at_k_std": f"ndcg@{results['summary']['ndcg_k']}_std",
        "recall_at_k_mean": f"recall@{results['summary']['recall_k']}_mean",
        "recall_at_k_std": f"recall@{results['summary']['recall_k']}_std",
        "sign_agreement_mean": "sign_mean",
        "sign_agreement_std": "sign_std",
        "runtime_s_mean": "method_runtime_s",
        "benchmark_runtime_s_mean": "eval_runtime_s",
        "method_abs_error_mean": "abs_error_mean",
        "selected_neurons_mean": "selected_neurons_mean",
    }
)
summary_df


,method,spearman_mean,spearman_std,ndcg@10_mean,ndcg@10_std,recall@10_mean,recall@10_std,sign_mean,sign_std,method_runtime_s,eval_runtime_s,abs_error_mean,selected_neurons_mean,n_images
0,NAA,0.593435,0.085930,0.827276,0.115396,0.610000,0.166032,0.799660,0.066065,2.826438,2.828128,10.982773,NaN,30
1,IG,0.550502,0.128270,0.829365,0.111829,0.656667,0.128279,0.835279,0.067284,5.465676,5.467781,0.899682,NaN,30
2,"Cheap-IG+[0,0.1]/k8000/zero",0.420839,0.106996,0.741999,0.145234,0.586667,0.128409,0.820331,0.089159,2.348335,2.349559,108.637131,8000.0,30


## Pairwise Win-Rate Matrix

In [9]:
pairwise_df = pd.DataFrame(results["summary"]["pairwise_win_rates"]).T
pairwise_df.index.name = "row_method"
pairwise_df


,IG,NAA,"Cheap-IG+[0,0.1]/k8000/zero"
row_method,,,
IG,0.000000,0.366667,0.933333
NAA,0.633333,0.000000,0.866667
"Cheap-IG+[0,0.1]/k8000/zero",0.066667,0.133333,0.000000


## Core / Oracle Diagnostics

In [10]:
core_summary = results["summary"]["core_summary"]
diagnostics_df = pd.DataFrame(
    {
        "metric": list(core_summary.keys()),
        "mean": [core_summary[key]["mean"] for key in core_summary],
        "std": [core_summary[key]["std"] for key in core_summary],
        "count": [core_summary[key]["count"] for key in core_summary],
    }
)
diagnostics_df


KeyError: 'count'

## Per-Image Metric Pivot

In [ ]:
rows_df = pd.DataFrame(results["rows"])
per_image_spearman = rows_df.pivot(index="image_name", columns="method_name", values="spearman")
per_image_spearman.head(10)


## Ключевые фигуры

In [ ]:
for key, title in [
    ("summary_metrics", "Summary Metrics"),
    ("distribution_metrics", "Distribution Metrics"),
    ("pairwise_spearman", "Pairwise Spearman Win-Rate"),
]:
    path = artifacts["figures"].get(key)
    if not path:
        continue
    display(Markdown(f"### {title}"))
    display(Image(filename=str(path)))


## Visual Preview Pages

In [ ]:
for section in artifacts["preview_sections"]:
    display(Markdown(f"### Preview Page {section['page_idx']}"))
    display(Image(filename=str(section["figure_path"])))
